In [1]:
# 1. Install dependencies in Colab if not present
# Set force_reinstall to True if you run into import/metadata errors
force_reinstall = False

try:
    import unsloth
    from unsloth import FastLanguageModel
    import torch
    import trl
    import peft
    import accelerate
    import bitsandbytes
    import datasets
    print("Unsloth and all dependencies are already installed and working.")
except (ImportError, Exception) as e:
    print(f"Dependency check failed ({e}). Installing/fixing dependencies...")
    force_reinstall = True

if force_reinstall:
    # First uninstall conflicting packages to avoid metadata corruption
    !pip uninstall -y unsloth unsloth-zoo unsloth_zoo vllm xformers trl peft accelerate bitsandbytes

    # Install official Unsloth (this automatically installs compatible versions of trl, peft, accelerate, and bitsandbytes)
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

    print("\n" + "="*80)
    print("IMPORTANT: Please restart your Colab runtime (Runtime > Restart session) now to load the newly installed packages!")
    print("="*80)


Dependency check failed (No module named 'unsloth'). Installing/fixing dependencies...
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-kwjz6n5z/unsloth_9af728f02e25436090bf9b287f64ca09
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-kwjz6n5z/unsloth_9af728f02e25436090bf9b287f64ca09
  Resolved https://github.com/unslothai/unsloth.git to commit 4098b4977eea87d31c1519a3f64bb03eb08e3ac3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 49.9 MB/s eta 0:00:00
 

In [2]:
# 2. Check GPU
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected! Fine-tuning requires a GPU. Please enable GPU in Colab settings.")
print(f"Using GPU: {torch.cuda.get_device_name(0)}")

Using GPU: NVIDIA A100-SXM4-40GB


In [3]:
# 3. Setup paths and directories
import os
import sys

is_colab = 'google.colab' in sys.modules
if is_colab:
    from google.colab import drive
    print("Mounting Google Drive...")
    drive.mount('/content/drive')

    # Drive Paths
    gdrive_root = "/content/drive/MyDrive/CryptoAgents_FT"
    data_dir = os.path.join(gdrive_root, "data")
    models_dir = os.path.join(gdrive_root, "models")

    train_path = os.path.join(data_dir, "train.jsonl")
    val_path = os.path.join(data_dir, "val.jsonl")
else:
    # Local paths for testing
    print("Running locally (testing mode)...")
    # In notebook, locate project root relative to notebook path
    train_path = "../data/train.jsonl"
    val_path = "../data/val.jsonl"
    models_dir = "../models"

if not os.path.exists(train_path):
    raise FileNotFoundError(f"Train dataset not found at {train_path}. Please upload it first!")

os.makedirs(models_dir, exist_ok=True)
os.makedirs(os.path.join(models_dir, "lora_adapter"), exist_ok=True)
os.makedirs(os.path.join(models_dir, "merged"), exist_ok=True)
os.makedirs(os.path.join(models_dir, "gguf"), exist_ok=True)

print(f"Train path: {train_path}")
print(f"Validation path: {val_path}")
print(f"Models directory: {models_dir}")

Mounting Google Drive...
Mounted at /content/drive
Train path: /content/drive/MyDrive/CryptoAgents_FT/data/train.jsonl
Validation path: /content/drive/MyDrive/CryptoAgents_FT/data/val.jsonl
Models directory: /content/drive/MyDrive/CryptoAgents_FT/models


In [4]:
# 4. Load base model Qwen3-4B in 4-bit
from unsloth import FastLanguageModel

max_seq_length = 4096
dtype = None # None for auto detection. Float16 for Tesla T4, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to save memory

print("Loading base Qwen3-4B model in 4-bit...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen3-4B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading base Qwen3-4B model in 4-bit...
==((====))==  Unsloth 2026.6.8: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.70k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/4.67k [00:00<?, ?B/s]

unsloth/qwen3-4b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [5]:
# 5. Setup LoRA Adapters
print("Setting up QLoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 32,
    lora_dropout = 0, # Dropout=0 is optimized for Unsloth
    bias = "none",
    use_gradient_checkpointing = "unsloth", # 4x memory savings
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Setting up QLoRA adapters...


Unsloth 2026.6.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [6]:
# 6. Load Dataset & Format ChatML
from datasets import load_dataset

print("Loading dataset splits...")
dataset = load_dataset("json", data_files={"train": train_path, "validation": val_path})

def format_prompts(batch):
    formatted_texts = []
    for messages in batch["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        formatted_texts.append(text)
    return {"text": formatted_texts}

dataset = dataset.map(format_prompts, batched=True)
print("ChatML mapping completed.")

Loading dataset splits...


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1152 [00:00<?, ? examples/s]

Map:   0%|          | 0/144 [00:00<?, ? examples/s]

ChatML mapping completed.


In [10]:
# 7. SFTTrainer Setup
from trl import SFTTrainer
from transformers import TrainingArguments

print("Initializing Trainer...")
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset["validation"],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 50,
        num_train_epochs = 5,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = os.path.join(models_dir, "outputs"),
        eval_strategy = "steps",
        eval_steps = 50,
        save_strategy = "no",
        report_to = "none"
    ),
)


Initializing Trainer...


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1152 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/144 [00:00<?, ? examples/s]

In [11]:
# 8. Start Fine-Tuning
print("Starting fine-tuning...")
trainer_stats = trainer.train()
print("Fine-tuning completed successfully!")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.2f} seconds")

Starting fine-tuning...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,152 | Num Epochs = 5 | Total steps = 720
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Step,Training Loss,Validation Loss
50,0.241110,0.244468
100,0.239597,0.240933
150,0.237237,0.238840
200,0.236538,0.237928
250,0.234360,0.237600
300,0.229854,0.236912
350,0.233898,0.237509
400,0.230376,0.236693
450,0.230948,0.236949
500,0.233310,0.236904


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Fine-tuning completed successfully!
Training time: 1086.46 seconds


In [12]:
# 9. Save LoRA Adapters
lora_save_path = os.path.join(models_dir, "lora_adapter")
print(f"Saving LoRA adapters to {lora_save_path}...")
model.save_pretrained(lora_save_path)
tokenizer.save_pretrained(lora_save_path)

Saving LoRA adapters to /content/drive/MyDrive/CryptoAgents_FT/models/lora_adapter...


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/CryptoAgents_FT/models/lora_adapter/tokenizer_config.json.


('/content/drive/MyDrive/CryptoAgents_FT/models/lora_adapter/tokenizer_config.json',
 '/content/drive/MyDrive/CryptoAgents_FT/models/lora_adapter/chat_template.jinja',
 '/content/drive/MyDrive/CryptoAgents_FT/models/lora_adapter/tokenizer.json')

In [13]:
# 10. Save Merged Model & Export GGUF
print("Merging adapters to 16bit base model...")
model.save_pretrained_merged(
    os.path.join(models_dir, "merged"),
    tokenizer,
    save_method = "merged_16bit"
)

print("Exporting merged model to GGUF format (q4_k_m quantization)...")
gguf_save_dir = os.path.join(models_dir, "gguf")
model.save_pretrained_gguf(
    gguf_save_dir,
    tokenizer,
    quantization_method = "q4_k_m"
)
print("Merged model and GGUF saved successfully!")

Merging adapters to 16bit base model...


config.json:   0%|          | 0.00/752 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/CryptoAgents_FT/models/merged/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/3.08G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:33<00:00, 16.56s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:40<00:00, 20.29s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/CryptoAgents_FT/models/gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b9744 (llama-b9744-bin-ubuntu-x64.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/drive/MyDrive/CryptoAgents_FT/models/gguf_gguf/qwen3-4b.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: